In [21]:
import glob
import os
import sys
import pandas as pd
from typing import Optional, Tuple, Union
from dataclasses import dataclass, field
import numpy as np
import logging
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


# Add project root to system path to allow src module imports
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

In [ ]:
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
console_handler.setFormatter(formatter)
log.addHandler(console_handler)


@dataclass
class EvalResult:

    tp: int = 0
    fp: int = 0
    fn: int = 0

    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    accuracy: float = 0.0

    dist: np.ndarray = field(default_factory=lambda: np.array([]))
    mean_dist: float = 0.0

    panoptic_quality: float = 0.0

    false_negatives: Tuple[int, ...] = field(default_factory=tuple)
    false_positives: Tuple[int, ...] = field(default_factory=tuple)
    matched_pairs: Tuple[Tuple[int, int], ...] = field(default_factory=tuple)


def points_matching(
    p1,
    p2,
    cutoff_distance=3,
    eps=1e-8,
    class_label_p1: Optional[int] = None,
    class_label_p2: Optional[int] = None,
):
    """finds matching that minimizes sum of mean squared distances"""
    if class_label_p1 is not None:
        p1 = p1[p1[:, -1] == class_label_p1][:, :-1]
    elif class_label_p2 is not None:
        p2 = p2[p2[:, -1] == class_label_p2][:, :-1]

    assert p1.shape[1] == p2.shape[1], "Last dimensions of p1, p2 must match"

    if len(p1) == 0 or len(p2) == 0:
        D = np.zeros((0, 0))
    else:
        D = cdist(p1, p2, metric="sqeuclidean")

    if D.size > 0:
        D[D > cutoff_distance**2] = 1e10 * (1 + D.max())

    i, j = linear_sum_assignment(D)
    valid = D[i, j] <= cutoff_distance**2

    i, j = i[valid], j[valid]

    res = EvalResult()

    tp = len(i)
    fp = len(p2) - tp
    fn = len(p1) - tp
    res.tp = tp
    res.fp = fp
    res.fn = fn

    # when there is no tp and we dont predict anything the accuracy should be 1 not 0
    tp_eps = tp + eps

    res.accuracy = tp_eps / (tp_eps + fp + fn) if tp_eps > 0 else 0
    res.precision = tp_eps / (tp_eps + fp) if tp_eps > 0 else 0
    res.recall = tp_eps / (tp_eps + fn) if tp_eps > 0 else 0
    res.f1 = (2 * tp_eps) / (2 * tp_eps + fp + fn) if tp_eps > 0 else 0
    res.dist = np.sqrt(D[i, j])
    res.mean_dist = float(np.mean(res.dist)) if len(res.dist) > 0 else 0.0

    pq_num = np.sum(cutoff_distance - res.dist) / cutoff_distance
    pq_den = tp_eps + fp / 2 + fn / 2
    res.panoptic_quality = float(pq_num / pq_den) if tp_eps > 0 else 0.0

    res.false_negatives = tuple(set(range(len(p1))).difference(set(i)))
    res.false_positives = tuple(set(range(len(p2))).difference(set(j)))
    res.matched_pairs = tuple(zip(i, j))
    return res


def points_matching_dataset(
    p1s,
    p2s,
    cutoff_distance=3,
    by_image=True,
    eps=1e-8,
    class_label_p1: Optional[int] = None,
    class_label_p2: Optional[int] = None,
):
    """
    by_image is True -> metrics are computed by image and then averaged
    by_image is False -> TP/FP/FN are aggregated and only then are metrics computed
    """
    stats = tuple(
        points_matching(
            p1,
            p2,
            cutoff_distance=cutoff_distance,
            eps=eps,
            class_label_p1=class_label_p1,
            class_label_p2=class_label_p2,
        )
        for p1, p2 in zip(p1s, p2s)
    )

    if by_image:
        res: EvalResult = EvalResult()
        for k, v in vars(stats[0]).items():
            if np.isscalar(v):
                setattr(res, k, float(np.mean([vars(s)[k] for s in stats])))
        return res
    else:
        res = EvalResult()
        res.tp = 0
        res.fp = 0
        res.fn = 0

        for s in stats:
            for k in ("tp", "fp", "fn"):
                setattr(res, k, getattr(res, k) + getattr(s, k))

        dists = np.concatenate([s.dist for s in stats])

        tp_eps = res.tp + eps
        res.accuracy = tp_eps / (tp_eps + res.fp + res.fn) if tp_eps > 0 else 0
        res.precision = tp_eps / (tp_eps + res.fp) if tp_eps > 0 else 0
        res.recall = tp_eps / (tp_eps + res.fn) if tp_eps > 0 else 0
        res.f1 = (2 * tp_eps) / (2 * tp_eps + res.fp + res.fn) if tp_eps > 0 else 0

        pq_num = np.sum(cutoff_distance - dists) / cutoff_distance
        pq_den = tp_eps + res.fp / 2 + res.fn / 2

        res.panoptic_quality = float(pq_num / pq_den) if tp_eps > 0 else 0.0
        res.mean_dist = float(np.mean(dists)) if len(dists) > 0 else 0.0
        return res

In [ ]:
def evaluate_model():
    ground_truth_df = pd.read_csv(config.TEST_CSV_PATH)
    prediction_df = pd.read_csv(config.NMS_RESULTS_CSV)

    p1 = ground_truth_df[["Motor axis 2", "Motor axis 1"]].to_numpy()
    p2 = prediction_df[["x_center", "y_center"]].to_numpy()

    results = points_matching(p1, p2, cutoff_distance=30, class_label_p1=None, class_label_p2=None)

    test = points_matching_dataset([p1], [p2], cutoff_distance=30, by_image=False)

    print(test)
    print(results)

    return results


def parse_csv_row(line: str):
    """Parse a CSV row into its components."""
    parts = line.strip().split(",")
    return {
        "tomo_id": parts[0],
        "slice": int(parts[1]),
        "confidence": float(parts[2]),
        "x_center": float(parts[3]),
        "y_center": float(parts[4]),
        "width": float(parts[5]),
        "height": float(parts[6]),
    }


def train_test_validation_split():
    """Generate train, validation, and test CSV files from the full training labels."""

    df = pd.read_csv(config.TRAIN_LABELS_PATH)

    print(f"Original set size: {len(df)}")

    # make blance between pozitive and negative samples %50 pozitive and %50 negative
    positive_samples = df[df["Number of motors"] > 0]
    negative_samples = df[df["Number of motors"] == 0]

    # Calculate split sizes
    total_positive = len(positive_samples)
    total_negative = len(negative_samples)

    # Calculate number of samples for each split
    train_pos = int(0.7 * total_positive)
    val_pos = int(0.2 * total_positive)

    train_neg = int(0.7 * total_negative)
    val_neg = int(0.2 * total_negative)

    # Shuffle and split positive samples
    positive_samples = positive_samples.sample(frac=1, random_state=1)
    train_positive = positive_samples.iloc[:train_pos]
    val_positive = positive_samples.iloc[train_pos : train_pos + val_pos]
    test_positive = positive_samples.iloc[train_pos + val_pos :]

    # Shuffle and split negative samples
    negative_samples = negative_samples.sample(frac=1, random_state=1)
    train_negative = negative_samples.iloc[:train_neg]
    val_negative = negative_samples.iloc[train_neg : train_neg + val_neg]
    test_negative = negative_samples.iloc[train_neg + val_neg :]

    # Combine positive and negative samples for each split
    train_df = pd.concat([train_positive, train_negative]).sample(frac=1, random_state=1)
    validation_df = pd.concat([val_positive, val_negative]).sample(frac=1, random_state=1)
    test_df = pd.concat([test_positive, test_negative]).sample(frac=1, random_state=1)

    # Save to CSV
    train_df.to_csv(config.TRAIN_CSV_PATH, index=False)
    validation_df.to_csv(config.VALIDATION_CSV_PATH, index=False)
    test_df.to_csv(config.TEST_CSV_PATH, index=False)


def find_all_predicted_tomo_id_attributes(tomo_id: str) -> Union[pd.DataFrame, None]:
    """Find all predicted attributes for a given tomo_id."""
    labels_df = pd.read_csv(config.DENORMALIZED_RESULTS)
    filtered_df = labels_df[labels_df["tomo_id"] == tomo_id]

    if filtered_df.empty:
        return None

    return filtered_df


def find_all_tomo_id_attributes(tomo_id: str) -> Union[pd.DataFrame, None]:
    """Find all attributes for a given tomo_id."""
    labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
    filtered_df = labels_df[labels_df["tomo_id"] == tomo_id]

    if filtered_df.empty:
        return None  # Return None if tomo_id not found

    return filtered_df


def find_tomo_id_slice_attributes(tomo_id, slice_num) -> Union[pd.DataFrame, None]:
    """Find attributes for a given tomo_id and slice_num."""
    print(config.TRAIN_LABELS_PATH)
    labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
    filtered_df = labels_df[(labels_df["tomo_id"] == tomo_id)]
    filtered_df = labels_df[(labels_df["Motor axis 0"] == slice_num)]

    if filtered_df.empty:
        return None  # Return None if tomo_id or slice_num not found

    return filtered_df


def find_width_height(tomo_id, slice_num):
    """Find width and height for a given tomo_id and slice_num."""
    attributes = find_tomo_id_slice_attributes(tomo_id, slice_num)

    if attributes is None or attributes.empty:
        print(f"No attributes found for tomo_id: {tomo_id}, slice_num: {slice_num}")
        return None, None

    width = attributes.iloc[0]["Array shape (axis 2)"]
    height = attributes.iloc[0]["Array shape (axis 1)"]
    print(f"Width: {width}, Height: {height}")
    return width, height


def parse_txt(file_path, f):
    """Parse a YOLO format .txt file and return its components."""
    file_name = os.path.basename(file_path)  # e.g. "image1.txt"
    file_id = os.path.splitext(file_name)[0]  # e.g. "image1"
    _, tomo_num, _, slice_idx = file_id.split("_")
    tomo_id = "tomo_" + str(tomo_num)
    _, confidence, x_center, y_center, width, height = f.readline().strip().split()
    return tomo_id, int(slice_idx), confidence, x_center, y_center, width, height


def rescale_letterbox():
    """Rescale and denormalize YOLO predictions to original image dimensions."""
    rows = []

    print(f"{config.RTDETR_RESULT_LABELS}/*.txt")
    txt_files = sorted(glob.glob(f"{config.RTDETR_RESULT_LABELS}/*.txt"))
    for file_path in txt_files:
        print(f"\n--- Reading: {file_path} ---")
        with open(file_path, "r", encoding="utf-8") as f:
            # print(parse_txt(file_path,f))
            tomo_id, slice_number, confidence, x_center, y_center, width, height = parse_txt(
                file_path, f
            )
            img_width, img_height = find_width_height(tomo_id, slice_number)

            if img_width is None or img_height is None:
                print(f"Skipping {tomo_id}, {slice_number} due to missing dimensions.")
                continue

            # denomarize
            x_center = float(x_center) * img_width
            y_center = float(y_center) * img_height
            width = float(width) * img_width
            height = float(height) * img_height

            # what about voxel spacing ?

            rows.append(
                {
                    "tomo_id": tomo_id,
                    "slice": slice_number,
                    "confidence": float(confidence),
                    "x_center": x_center,
                    "y_center": y_center,
                    "width": width,
                    "height": height,
                }
            )

    df = pd.DataFrame(rows)
    df.to_csv(config.DENORMALIZED_RESULTS, index=False)


#
# find_all_tomoId_attributes("tomo_00e047")
# evaluate_model()
train_test_validation_split()

Original set size: 737
